In [1]:
#!pip install eurostat

In [2]:
import pandas as pd
import eurostat

code = "prc_ppp_ind_1"

# 1. What are the dimensions called?
pars = eurostat.get_pars(code)
print(pars)

# 2. For each dimension, show its codes and labels
for p in pars:
    print("\n==", p, "==")
    try:
        d = eurostat.get_dic(code, p)
        print(pd.DataFrame(d, columns=["code", "label"]).head(60).to_string())
    except Exception as e:
        print("could not load:", e)

['freq', 'indic_ppp', 'ppp_cat18', 'geo']

== freq ==
   code                    label
0     P             Pluri-annual
1     A                   Annual
2     S  Half-yearly, semesterly
3     Q                Quarterly
4     M                  Monthly
5     W                   Weekly
6     B    Daily - business week
7     D                    Daily
8     H                   Hourly
9     I   Irregular / A-periodic
10  NAP           Not applicable
11   A3                Triannual

== indic_ppp ==
                     code                                                                              label
0                   TOTAL                                                                              Total
1           PPP_EU27_2020                                            Purchasing power parities (EU27_2020=1)
2                PPP_EU28                                                 Purchasing power parities (EU28=1)
3           PPP_EU27_2007                                       

In [3]:
code = "prc_ppp_ind_1"
raw = eurostat.get_data_df(code)
print(raw.shape)
print(raw.columns.tolist()[:8])   # see how the columns are named

# The geo column is often named "geo\TIME_PERIOD"; find it
geo_col = [c for c in raw.columns if c.startswith("geo")][0]

# Keep price level indices (EU27 = 100), annual, for the categories we want
cats = ["A01", "A010101", "A0111", "A0107", "A010703", "A0109"]
pli = raw[(raw["indic_ppp"] == "PLI_EU27_2020") &
          (raw["freq"] == "A") &
          (raw["ppp_cat18"].isin(cats))].copy()

# Which of your countries are present?
wanted = {"FR":"France","ES":"Spain","EL":"Greece","UK":"United Kingdom",
          "CH":"Switzerland","TR":"Turkey","US":"United States","JP":"Japan",
          "AU":"Australia","NZ":"New Zealand","CA":"Canada",
          "ZA":"South Africa","TH":"Thailand","CN":"China"}
present = set(pli[geo_col].unique())
for k, v in wanted.items():
    print(f"{v:15s} {k}  ->", "PRESENT" if k in present else "missing")

# For present countries: how many categories and which years?
year_cols = [c for c in pli.columns if c[:2] in ("19", "20")]
for k in wanted:
    if k in present:
        sub = pli[pli[geo_col] == k]
        ok = sub[year_cols].notna().any()
        years = [c for c in year_cols if ok[c]]
        print(k, "| categories:", sub["ppp_cat18"].nunique(),
              "| years:", years[0], "to", years[-1])

pli.to_csv("eurostat_pli_raw.csv", index=False)

(19200, 35)
['freq', 'indic_ppp', 'ppp_cat18', 'geo\\TIME_PERIOD', '1995', '1996', '1997', '1998']
France          FR  -> PRESENT
Spain           ES  -> PRESENT
Greece          EL  -> PRESENT
United Kingdom  UK  -> PRESENT
Switzerland     CH  -> PRESENT
Turkey          TR  -> PRESENT
United States   US  -> PRESENT
Japan           JP  -> PRESENT
Australia       AU  -> missing
New Zealand     NZ  -> missing
Canada          CA  -> missing
South Africa    ZA  -> missing
Thailand        TH  -> missing
China           CN  -> missing
FR | categories: 6 | years: 1995 to 2025
ES | categories: 6 | years: 1995 to 2025
EL | categories: 6 | years: 1995 to 2025
UK | categories: 1 | years: 1995 to 2025
CH | categories: 6 | years: 1995 to 2025
TR | categories: 6 | years: 1995 to 2025
US | categories: 1 | years: 1995 to 2025
JP | categories: 1 | years: 1995 to 2025
